# Episode 4 — `vmap`, `scan`, and Vectorization

**Instructor notebook** · run top-to-bottom before recording.

Batch with `vmap`, replace Python loops with `lax.scan`, and see how XLA vectorizes.

| | |
|---|---|
| **Chapter** | 1.4 · Part I — Pure JAX |
| **Prereq** | Episodes 1–3 |
| **Next** | Episode 5 — pytrees and SGD |

**JAX docs:** [`jax.vmap`](https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html) · [`jax.lax.scan`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.scan.html) · [JAX 101 — `scan`](https://jax.readthedocs.io/en/latest/jax-101.html#writing-a-loop-using-scan)


In [ ]:
import timeit

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import lax
from jax import make_jaxpr

## XLA and vectorization (preview)

Episode 2 traced ops into a **jaxpr**, then XLA lowered it to fused kernels. **`vmap`** adds a batch axis to that program so XLA can issue wide SIMD/GPU matmuls instead of a Python loop over samples. **`scan`** does the same for loops over time — one compiled recurrence instead of per-step Python.

## Single-sample forward, then `vmap`

In [ ]:
def forward(params, x):
    return jnp.tanh(x @ params["w"] + params["b"])


params = {"w": jr.normal(jr.key(0), (8, 4)) * 0.1, "b": jnp.zeros(4)}
B = 16
x_batch = jr.normal(jr.key(1), (B, 8))

y_loop = jnp.stack([forward(params, x_batch[i]) for i in range(B)])
batched_forward = jax.vmap(forward, in_axes=(None, 0))
y_vmap = batched_forward(params, x_batch)
print("vmap matches loop:", jnp.allclose(y_loop, y_vmap))
print("shape:", y_vmap.shape)

## `vmap` vs handwritten loop

In [ ]:
def time_ms(fn, *args, warmup=2, repeat=5):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))
    return min(
        timeit.repeat(lambda: jax.block_until_ready(fn(*args)), number=1, repeat=repeat)
    ) * 1000


big_x = jr.normal(jr.key(2), (512, 8))
loop_fn = lambda p, x: jnp.stack([forward(p, x[i]) for i in range(x.shape[0])])
vmap_fn = jax.jit(jax.vmap(forward, in_axes=(None, 0)))
print(f"loop: {time_ms(loop_fn, params, big_x):.2f} ms")
print(f"vmap: {time_ms(vmap_fn, params, big_x):.2f} ms")

## `jit` ∘ `vmap` — batched kernel in jaxpr

In [ ]:
jitted_batched = jax.jit(jax.vmap(forward, in_axes=(None, 0)))
print(make_jaxpr(jitted_batched)(params, x_batch))

## Prefix sum — Python `for` vs `lax.scan`

Inclusive prefix: y_t = sum of x_i for i <= t. Compare a Python loop (no `jit`) to `scan` threading carry state.

In [ ]:
def prefix_sum_loop(x):
    out = []
    total = 0.0
    for xi in x:
        total = total + xi
        out.append(total)
    return jnp.array(out)


def prefix_sum_scan(x):
    def step(carry, xi):
        new_carry = carry + xi
        return new_carry, new_carry

    _, ys = lax.scan(step, 0.0, x)
    return ys


x = jnp.ones(100_000)
y_loop = prefix_sum_loop(x)
y_scan = prefix_sum_scan(x)
print("match cumsum:", jnp.allclose(y_scan, jnp.cumsum(x)))
print("match loop:  ", jnp.allclose(y_loop, y_scan))

## Time loop vs `scan`

In [ ]:
print(f"loop: {time_ms(prefix_sum_loop, x):.2f} ms")
print(f"scan: {time_ms(prefix_sum_scan, x):.2f} ms")

> **Key insight:** `vmap` vectorizes over batch without rewriting code; `scan` does the same over time. Later episodes batch with a leading tensor dimension instead of `vmap`.

---
## Exercise

1. `vmap` a single-sample forward; verify against a Python loop.
2. Time `vmap` vs loop on a large batch.
3. Implement inclusive prefix sum with `scan`; compare to a Python `for` loop.

*(Solution below.)*

In [ ]:
print('vmap out:', y_vmap.shape, '| prefix sum ok:', jnp.allclose(y_scan, jnp.cumsum(x)))

**Next:** Episode 5 — pytrees and SGD.